<a href="https://colab.research.google.com/github/bauhaushm/TP-Delito_en_AMBA/blob/bauhaushm/TP_Delito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabajo Práctico Grupal - Laboratorio de Métodos Cuantitativos
**Universidad de Buenos Aires - Facultad de Ciencias Económicas**  
**Tecnicatura Universitaria en Gestión y Análisis de Datos**  
**Segundo Cuatrimestre 2026**

---

### Integrantes del Grupo:
* **Chietino Luna, Micaela** - Registro: 921785 - Usuario GitHub: `@mchietino`
* **Iandoli, Franco** - Registro: 921507 - Usuario GitHub: `@Franco-Iandoli`
* **Ortiz, Naiara** - Registro: 921844 - Usuario GitHub: `@naiaraortizrodriguez`
* **Peralta, Santiago** - Registro: 921431 - Usuario GitHub: `@SantiagoNPeralta`
* **Turrillo, Bruno** - Registro: 921414 - Usuario GitHub: `@bauhaushm`

---

## 1. Introducción y Pregunta de Investigación

Nuestro grupo se focalizará en la temática del delito y su relación con aspectos sociales que pueden influir en él.

**a) Pregunta de investigación:**

> *"¿Cómo se comparan los patrones delictivos entre los partidos del AMBA, y de qué manera varían el tipo y la frecuencia de los delitos según las condiciones socioeconómicas y habitacionales al interior de las comunas porteñas?"*

**b) ¿Por qué vale la pena responderla?**
[Completar: qué decisión, fenómeno o problema ilumina. Por ejemplo, que el AMBA concentra gran parte de la población del país y que las políticas de seguridad requieren saber qué delitos predominan en cada territorio y si la brecha entre CABA y el conurbano se achica o se amplía.]

**c) ¿Qué respuesta encontramos?**
[Completar al finalizar el análisis, volviendo a esta celda con los hallazgos concretos.]

### 1.1 Fuente de datos

Utilizamos como fuente principal el Sistema Nacional de Información Criminal (SNIC), del Ministerio de Seguridad de la Nación. La base está disponible en https://www.argentina.gob.ar/seguridad/estadisticascriminales/bases-de-datos y contiene distintos datasets, de los cuales elegimos **"SNIC - Departamentos. Anual. Años 2000-2025"**.

Este dataset cubre todo el país, por lo que lo primero que hacemos es recortarlo al **Área Metropolitana de Buenos Aires (AMBA)**, que es el territorio que nos interesa para responder la pregunta de investigación.

## 2. Carga y exploración inicial de los datos

## 2.1 Importación de librerías

Importamos las librerías que vamos a usar: `pandas` para manipular los archivos, `matplotlib` y `seaborn` para los gráficos, `numpy` para operaciones matemáticas, `colorsys` para definir colores de las barras (convierte colores de HLS a RGB), y `requests` e `io` para descargar el archivo de forma controlada.


In [54]:
import io
import colorsys
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### 2.2 Carga del dataset con fuente de respaldo y recorte al AMBA

En esta celda cargamos el dataset del SNIC con una función que prueba dos fuentes en orden:

1. **URL oficial del SNIC (Ministerio de Seguridad).** Es la fuente primaria, porque garantiza trabajar con la versión original y más actualizada de los datos.
2. **Copia del recorte ya filtrado (AMBA, 2017-2025) del mismo dataset, almacenada en el repositorio de GitHub del grupo.** Si la página oficial no responde (caída del sitio, error de red o tiempo de espera agotado), la función usa esta copia como respaldo.

Elegimos esta estructura por dos motivos. Primero, **reproducibilidad**: el notebook se ejecuta completo sin intervención manual, tanto en nuestras sesiones de Colab como cuando lo revise otra persona, aunque el servidor del SNIC esté temporalmente fuera de servicio. Segundo, **transparencia**: la función informa por pantalla de cuál de las dos fuentes se cargaron los datos, de modo que queda registro de qué versión se utilizó. Usamos la librería `requests` con un límite de tiempo (`timeout`) para evitar que la ejecución quede detenida indefinidamente esperando una respuesta.

Del archivo original pedimos solo seis columnas: "Año", "Provincia", "Departamento", "Tipo de delito", "Cantidad de hechos" y "Tasa de hechos" (cada 100.000 habitantes). Luego aplicamos dos filtros:

- **Período 2017-2025** (los últimos 9 años), para centrar el análisis en el escenario reciente y reducir el tamaño del dataset.
- **Territorio AMBA:** las 15 comunas de la Ciudad Autónoma de Buenos Aires y los 24 partidos del Gran Buenos Aires (conurbano bonaerense). Para evitar confundir partidos con nombres repetidos en otras provincias (como San Martín o Avellaneda), exigimos que coincidan a la vez la provincia y el departamento.

In [42]:
# 1. Fuentes de datos
URL_SNIC = "https://cloud-snic.minseg.gob.ar/Bases/SNIC/snic-departamentos-anual.xlsx"
URL_REPO = "https://raw.githubusercontent.com/bauhaushm/TP-Delito_en_AMBA/bauhaushm/snic_amba_2017_2025.csv"
columnas = [0, 2, 4, 6, 7, 12]  # Año, Provincia, Departamento, Tipo de delito, Cantidad de hechos, Tasa

def cargar_snic():
    # Intento 1: URL oficial
    try:
        r = requests.get(URL_SNIC, timeout=30)
        r.raise_for_status()
        print("Datos cargados desde la URL oficial del SNIC.")
        return pd.read_excel(io.BytesIO(r.content), usecols=columnas)
    except Exception as e:
        print(f"No se pudo usar la URL oficial ({type(e).__name__}). Usando copia del repositorio...")
    # Intento 2: copia reducida (AMBA, 2017-2025) en el repositorio
    r = requests.get(URL_REPO, timeout=60)
    r.raise_for_status()
    print("Datos cargados desde la copia reducida en el repositorio de GitHub.")
    return pd.read_csv(io.BytesIO(r.content))

# 2. Cargar el dataset nacional
df = cargar_snic()

# 3. Definir el territorio AMBA
partidos_gba = ["Almirante Brown", "Avellaneda", "Berazategui", "Esteban Echeverría",
    "Ezeiza", "Florencio Varela", "General San Martín", "Hurlingham", "Ituzaingó",
    "José C. Paz", "La Matanza", "Lanús", "Lomas de Zamora", "Malvinas Argentinas",
    "Merlo", "Moreno", "Morón", "Quilmes", "San Fernando", "San Isidro", "San Miguel",
    "Tigre", "Tres de Febrero", "Vicente López"]

es_periodo = df["anio"].between(2017, 2025)
es_caba = df["provincia_nombre"] == "Ciudad Autónoma de Buenos Aires"
es_conurbano = (df["provincia_nombre"] == "Buenos Aires") & (df["departamento_nombre"].isin(partidos_gba))

# 4. Filtrar período y territorio
df_amba = df[es_periodo & (es_caba | es_conurbano)].copy().reset_index(drop=True)

# 5. Verificar el resultado
print(f"El dataset del AMBA contiene {len(df_amba)} filas y {len(df_amba.columns)} columnas.")
print("Años incluidos:", sorted(df_amba["anio"].unique().tolist()))

# 6. Guardar el dataset resultante
df_amba.to_csv("snic_amba_2017_2025.csv", index=False)

Datos cargados desde la URL oficial del SNIC.
El dataset del AMBA contiene 21345 filas y 6 columnas.
Años incluidos: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


### 2.3 Primeras y últimas filas

Mostramos las primeras y últimas filas con `.head()` y `.tail()` para tener una primera impresión del contenido y confirmar que el orden y los valores tienen sentido.

In [43]:
display(df_amba.head())
display(df_amba.tail())

,anio,provincia_nombre,departamento_nombre,codigo_delito_snic_nombre,cantidad_hechos,tasa_hechos
0,2017,Ciudad Autónoma de Buenos Aires,Comuna 1,Homicidios dolosos,25,9.870850
1,2017,Ciudad Autónoma de Buenos Aires,Comuna 1,Abusos sexuales con acceso carnal (violaciones),14,5.527676
2,2017,Ciudad Autónoma de Buenos Aires,Comuna 1,Otros delitos contra la integridad sexual,190,75.018458
3,2017,Ciudad Autónoma de Buenos Aires,Comuna 1,Delitos contra el estado civil,0,0.000000
4,2017,Ciudad Autónoma de Buenos Aires,Comuna 1,Amenazas,1074,424.051707


,anio,provincia_nombre,departamento_nombre,codigo_delito_snic_nombre,cantidad_hechos,tasa_hechos
21340,2025,Buenos Aires,Vicente López,Lesiones dolosas,435,155.000641
21341,2025,Buenos Aires,Vicente López,Lesiones culposas en Accidentes Viales,681,242.656177
21342,2025,Buenos Aires,Vicente López,Lesiones culposas por otros hechos,13,4.632203
21343,2025,Buenos Aires,Vicente López,Otros delitos contra las personas,30,10.689699
21344,2025,Buenos Aires,Vicente López,Delitos contra el honor,0,0.000000


### 2.4 Estructura del dataset

Describimos las dimensiones, las columnas, sus tipos de dato (`.dtypes`) y la cobertura temporal y geográfica, para verificar que cada variable tenga el tipo adecuado (por ejemplo, el año como entero y la tasa como número decimal).

In [44]:
print("Dimensiones (filas, columnas):", df_amba.shape)
print("\nColumnas y tipos de dato:")
print(df_amba.dtypes)
print("\nCobertura temporal:", df_amba["anio"].min(), "a", df_amba["anio"].max())
print("Comunas y partidos distintos:", df_amba["departamento_nombre"].nunique())
print("Tipos de delito distintos:", df_amba["codigo_delito_snic_nombre"].nunique())

Dimensiones (filas, columnas): (21345, 6)

Columnas y tipos de dato:
anio                           int64
provincia_nombre              object
departamento_nombre           object
codigo_delito_snic_nombre     object
cantidad_hechos                int64
tasa_hechos                  float64
dtype: object

Cobertura temporal: 2017 a 2025
Comunas y partidos distintos: 40
Tipos de delito distintos: 66


### 2.5 Calidad de los datos

Analizamos la presencia de valores nulos (`.isnull().sum()`), filas duplicadas y estadísticas descriptivas (`.describe()`). También calculamos qué proporción de las filas tiene cero hechos, porque en el SNIC un cero puede significar que no hubo casos, y eso condiciona cómo interpretamos las tasas y los promedios.

In [45]:
print("Valores nulos por columna:")
print(df_amba.isnull().sum())
print("\nFilas duplicadas:", df_amba.duplicated().sum())
print(f"\nProporción de filas con cero hechos: {(df_amba['cantidad_hechos'] == 0).mean():.1%}")
df_amba.describe()

Valores nulos por columna:
anio                           0
provincia_nombre               0
departamento_nombre            0
codigo_delito_snic_nombre      0
cantidad_hechos                0
tasa_hechos                  534
dtype: int64

Filas duplicadas: 0

Proporción de filas con cero hechos: 42.2%


,anio,cantidad_hechos,tasa_hechos
count,21345.000000,21345.000000,20811.000000
mean,2021.170063,207.841368,67.360475
std,2.609329,736.560238,232.586402
min,2017.000000,0.000000,0.000000
25%,2019.000000,0.000000,0.000000
50%,2021.000000,3.000000,1.282462
75%,2023.000000,73.000000,22.706979
max,2025.000000,22881.000000,4053.030001


### 2.6 Diagnóstico de valores nulos en la tasa

Antes de decidir cómo tratar los nulos de `tasa_hechos`, analizamos cuántos hay en el AMBA y cómo se distribuyen por año, por comuna o partido y por cantidad de hechos.

In [46]:
nulos_tasa = df_amba[df_amba["tasa_hechos"].isna()]
print(f"Nulos en tasa_hechos: {len(nulos_tasa)} ({len(nulos_tasa) / len(df_amba):.2%})")
print("\nPor año:")
print(nulos_tasa.groupby("anio").size())
print("\nPor comuna/partido (top 10):")
print(nulos_tasa.groupby("departamento_nombre").size().sort_values(ascending=False).head(10))
print("\nCantidad de hechos en las filas con tasa nula:")
print(nulos_tasa["cantidad_hechos"].describe())

Nulos en tasa_hechos: 534 (2.50%)

Por año:
anio
2017    56
2018    56
2019    56
2020    56
2021    56
2022    56
2023    66
2024    66
2025    66
dtype: int64

Por comuna/partido (top 10):
departamento_nombre
Departamento sin determinar    534
dtype: int64

Cantidad de hechos en las filas con tasa nula:
count      534.000000
mean       236.765918
std        811.529762
min          0.000000
25%          1.000000
50%          9.500000
75%         96.750000
max      10000.000000
Name: cantidad_hechos, dtype: float64


### 2.7 Exclusión de los hechos sin comuna asignada

Los registros con `departamento_nombre` igual a "Departamento sin determinar" corresponden a hechos que el SNIC no pudo atribuir a una comuna o partido. Representan todos los nulos de `tasa_hechos`, porque sin territorio no hay población de referencia para calcular la tasa. Como nuestra pregunta compara patrones entre comunas y partidos, estos registros no son utilizables. Antes de excluirlos, cuantificamos cuántos hechos representan para declarar el efecto como limitación.

In [47]:
etiqueta = "Departamento sin determinar"
sd = df_amba[df_amba["departamento_nombre"] == etiqueta]

print("Provincia de origen de las filas sin determinar:")
print(sd["provincia_nombre"].value_counts())

hechos_sd = sd["cantidad_hechos"].sum()
hechos_amba = df_amba["cantidad_hechos"].sum()
caba = df_amba["provincia_nombre"] == "Ciudad Autónoma de Buenos Aires"
hechos_caba = df_amba.loc[caba, "cantidad_hechos"].sum()

print(f"\nHechos sin comuna asignada: {hechos_sd:,.0f}")
print(f"Porcentaje del total AMBA: {hechos_sd / hechos_amba:.2%}")
print(f"Porcentaje del total de CABA: {hechos_sd / hechos_caba:.2%}")

Provincia de origen de las filas sin determinar:
provincia_nombre
Ciudad Autónoma de Buenos Aires    534
Name: count, dtype: int64

Hechos sin comuna asignada: 126,433
Porcentaje del total AMBA: 2.85%
Porcentaje del total de CABA: 7.08%


In [48]:
df_amba = df_amba[df_amba["departamento_nombre"] != etiqueta].reset_index(drop=True)

print(f"Filas restantes: {len(df_amba)}")
print("Nulos en tasa_hechos:", df_amba["tasa_hechos"].isna().sum())
print("Nulos totales por columna:")
print(df_amba.isnull().sum())

Filas restantes: 20811
Nulos en tasa_hechos: 0
Nulos totales por columna:
anio                         0
provincia_nombre             0
departamento_nombre          0
codigo_delito_snic_nombre    0
cantidad_hechos              0
tasa_hechos                  0
dtype: int64


### 2.8 Verificación del recorte

Comprobamos que el filtro haya capturado las 15 comunas de CABA y los 24 partidos del conurbano. Si algún partido figura como faltante, es probable que el SNIC lo nombre distinto (abreviaturas, tildes, mayúsculas) y haya que ajustar la lista.

In [49]:
dep_caba = sorted(df_amba.loc[df_amba["provincia_nombre"] == "Ciudad Autónoma de Buenos Aires", "departamento_nombre"].unique())
dep_conurbano = sorted(df_amba.loc[df_amba["provincia_nombre"] == "Buenos Aires", "departamento_nombre"].unique())

print(f"Comunas de CABA encontradas: {len(dep_caba)} (esperadas: 15)")
print(f"Partidos del conurbano encontrados: {len(dep_conurbano)} (esperados: 24)")
print("Partidos faltantes:", sorted(set(partidos_gba) - set(dep_conurbano)))

Comunas de CABA encontradas: 15 (esperadas: 15)
Partidos del conurbano encontrados: 24 (esperados: 24)
Partidos faltantes: []


### 2.9 Diagnóstico de los tipos de delito

El SNIC pasa de 56 a 66 tipos de delito en 2023. Identificamos cuáles son los tipos nuevos y comprobamos que no existan categorías agregadas ("Total") que puedan inflar las sumas. Además comparamos los hechos de los tipos comunes y de los nuevos por año, para evaluar si el cambio de clasificación afecta la comparabilidad de la serie.

In [50]:
tipos_por_anio = {a: set(g["codigo_delito_snic_nombre"]) for a, g in df_amba.groupby("anio")}
tipos_comunes = set.intersection(*tipos_por_anio.values())
tipos_nuevos = tipos_por_anio[2025] - tipos_comunes

print(f"Tipos presentes en todos los años: {len(tipos_comunes)}")
print(f"Tipos que aparecen desde 2023: {len(tipos_nuevos)}")
for t in sorted(tipos_nuevos):
    print("  -", t)

print("\nTipos que contienen 'total':", [t for t in df_amba['codigo_delito_snic_nombre'].unique() if "total" in t.lower()])

es_comun = df_amba["codigo_delito_snic_nombre"].isin(tipos_comunes)
resumen = pd.DataFrame({
    "hechos_tipos_comunes": df_amba[es_comun].groupby("anio")["cantidad_hechos"].sum(),
    "hechos_tipos_nuevos": df_amba[~es_comun].groupby("anio")["cantidad_hechos"].sum()
}).fillna(0)
resumen

Tipos presentes en todos los años: 56
Tipos que aparecen desde 2023: 10
  - Abuso sexual agravado
  - Abuso sexual simple
  - Acceso ilegal a sistemas informáticos y daños informáticos
  - Ciberdelitos sexuales vinculados a menores
  - Daños (no incluye informáticos)
  - Estafas y defraudaciones (no incluye virtuales) y usura
  - Estafas y defraudaciones asistidas virtualmente
  - Extorsiones
  - Secuestros extorsivos
  - Tentativa de abuso sexual con acceso carnal

Tipos que contienen 'total': []


,hechos_tipos_comunes,hechos_tipos_nuevos
anio,,
2017,414286,0.0
2018,430558,0.0
2019,518968,0.0
2020,401475,0.0
2021,472926,0.0
2022,494481,0.0
2023,477888,48327.0
2024,488345,61343.0
2025,455144,46200.0


### 2.10 Exclusión de los tipos de delito incorporados desde 2023

Para que la serie 2017-2025 sea comparable, nos quedamos solo con los 56 tipos de delito presentes en todos los años y excluimos los 10 que el SNIC incorporó en 2023. De lo contrario, los totales anuales mostrarían un aumento artificial a partir de ese año que reflejaría un cambio de clasificación y no un cambio real en el delito. Cuantificamos cuántos hechos se dejan afuera para declararlo como limitación.

In [51]:
filas_antes = len(df_amba)
hechos_antes = df_amba["cantidad_hechos"].sum()

df_amba = df_amba[df_amba["codigo_delito_snic_nombre"].isin(tipos_comunes)].reset_index(drop=True)

hechos_despues = df_amba["cantidad_hechos"].sum()
print(f"Filas: {filas_antes} -> {len(df_amba)}")
print(f"Hechos excluidos: {hechos_antes - hechos_despues:,.0f} ({1 - hechos_despues / hechos_antes:.2%} del total 2017-2025)")
print("\nTipos de delito por año:")
print(df_amba.groupby("anio")["codigo_delito_snic_nombre"].nunique())

Filas: 20811 -> 19641
Hechos excluidos: 155,870 (3.62% del total 2017-2025)

Tipos de delito por año:
anio
2017    56
2018    56
2019    56
2020    56
2021    56
2022    56
2023    56
2024    56
2025    56
Name: codigo_delito_snic_nombre, dtype: int64


### 2.11 Visualización del dataframe del AMBA

Visualizamos el dataframe resultante. Cada fila representa la cantidad de hechos y la tasa por 100.000 habitantes de un tipo de delito, en una comuna o partido y un año determinados.

In [52]:
df_amba

,anio,provincia_nombre,departamento_nombre,codigo_delito_snic_nombre,cantidad_hechos,tasa_hechos
0,2017,Ciudad Autónoma de Buenos Aires,Comuna 1,Homicidios dolosos,25,9.870850
1,2017,Ciudad Autónoma de Buenos Aires,Comuna 1,Abusos sexuales con acceso carnal (violaciones),14,5.527676
2,2017,Ciudad Autónoma de Buenos Aires,Comuna 1,Otros delitos contra la integridad sexual,190,75.018458
3,2017,Ciudad Autónoma de Buenos Aires,Comuna 1,Delitos contra el estado civil,0,0.000000
4,2017,Ciudad Autónoma de Buenos Aires,Comuna 1,Amenazas,1074,424.051707
...,...,...,...,...,...,...
19636,2025,Buenos Aires,Vicente López,Lesiones dolosas,435,155.000641
19637,2025,Buenos Aires,Vicente López,Lesiones culposas en Accidentes Viales,681,242.656177
19638,2025,Buenos Aires,Vicente López,Lesiones culposas por otros hechos,13,4.632203
19639,2025,Buenos Aires,Vicente López,Otros delitos contra las personas,30,10.689699


### 2.12 Hallazgos de la exploración

- **Estructura:** el recorte del AMBA, para 2017-2025, tenía 21.345 filas y 6 columnas, con los tipos adecuados (año y cantidad de hechos como enteros, tasa como decimal). Tras la exclusión de los hechos sin comuna asignada y de los tipos de delito incorporados desde 2023, quedan [N] filas, correspondientes a 15 comunas de CABA y 24 partidos del conurbano.
- **Valores nulos:** el único nulo estaba en `tasa_hechos` (534 casos, 2,50%), todos en registros con "Departamento sin determinar", es decir, hechos que el SNIC no pudo atribuir a una comuna y para los que no existe población de referencia. Los excluimos, ya que no son utilizables para comparar territorios. Corresponden íntegramente a CABA y representan el 2,85% de los hechos del AMBA y el 7,08% de los de CABA, por lo que los totales de CABA quedan levemente subestimados y las comparaciones se hacen a nivel de comuna y partido. Tras la exclusión no quedan nulos en ninguna columna.
- **Cobertura de tipos de delito:** el SNIC pasa de 56 a 66 tipos desde 2023 (se incorporan, entre otros, ciberdelitos, extorsiones, secuestros extorsivos y estafas asistidas virtualmente). Esos 10 tipos nuevos representan entre el 9% y el 11% de los hechos de 2023-2025 y generarían un salto artificial en la serie temporal. Para garantizar la comparabilidad entre años, el análisis se restringe a los 56 tipos presentes en todo el período, lo que deja fuera el 3,6% de los hechos del total 2017-2025.
- **Duplicados:** no se encontraron filas duplicadas.
- **Ceros:** antes de las exclusiones, el 42,2% de las filas registraba cero hechos, ya que muchos tipos de delito son poco frecuentes en la mayoría de los territorios. Los conservamos, porque un cero informa que no hubo casos.
- **Estadísticos descriptivos:** antes de las exclusiones, la distribución de `tasa_hechos` era muy asimétrica: la mediana (alrededor de 1,3) era mucho menor que la media (alrededor de 67) y el máximo superaba las 4.000 cada 100.000 habitantes. Por eso priorizaremos totales agregados y comparaciones relativas, y no promedios simples entre tipos de delito.

## 3. Transformaciones y resumen estadístico

### 3.1 Creación de la variable `zona`

Para comparar CABA con el conurbano creamos una columna derivada `zona` que clasifica cada registro según su provincia. Esta variable es la base de todas las comparaciones que siguen.

In [55]:
df_amba["zona"] = np.where(df_amba["provincia_nombre"] == "Ciudad Autónoma de Buenos Aires", "CABA", "Conurbano")
df_amba["zona"].value_counts()

,count
zona,
Conurbano,12096
CABA,7545


### 3.2 Hechos delictivos totales por zona y año

Agrupamos por zona y año y sumamos la cantidad de hechos para ver el volumen de delitos registrado en CABA y en el conurbano y cómo evolucionó.

In [56]:
hechos_zona_anio = df_amba.groupby(["anio", "zona"])["cantidad_hechos"].sum().unstack()
hechos_zona_anio

zona,CABA,Conurbano
anio,,
2017,176874,237412
2018,169763,260795
2019,195212,323756
2020,133138,268337
2021,172026,300900
2022,189465,305016
2023,183014,294874
2024,187558,300787
2025,177271,277873


**Interpretación:** el conurbano registra más hechos que CABA en todos los años del período, entre 1,3 y 2,0 veces más (1,3 en 2017, 2,0 en 2020 y 1,6 en 2025). Esto es esperable en parte por su mayor población, por lo que la comparación entre zonas se hace con la tasa (sección 3.3) y no con el volumen.

En la evolución, ambas zonas alcanzan su máximo en 2019 y caen en 2020 (CABA 32%, conurbano 17%), un descenso probablemente asociado a las restricciones a la circulación por la pandemia; la mayor caída de CABA amplía la brecha ese año. Desde 2021 las dos se recuperan y la relación conurbano/CABA se estabiliza en torno a 1,6. Entre 2017 y 2025 los hechos en CABA se mantienen prácticamente iguales (+0,2%), mientras que el conurbano crece 17%, aunque ese aumento se concentra hasta 2019 (+36% respecto de 2017) y luego se modera. En 2025 ambas zonas descienden respecto de 2024 (CABA 5,5%, conurbano 7,6%).

Cabe recordar que son hechos registrados, que dependen de las denuncias, y que los totales de CABA están subestimados un 7% por los hechos sin comuna asignada, lo que no altera el sentido de la comparación.

### 3.3 Tasa total de delitos por comuna y partido

Calculamos la tasa total de cada territorio sumando las tasas de todos los tipos de delito (son categorías excluyentes con la misma población de referencia) y promediándola entre 2017 y 2025. Esto permite ordenar a las 39 unidades por nivel de delito relativo a su población.

In [57]:
tasa_total = (df_amba.groupby(["zona", "departamento_nombre", "anio"])["tasa_hechos"]
              .sum().reset_index(name="tasa_total"))
ranking = (tasa_total.groupby(["zona", "departamento_nombre"])["tasa_total"]
           .mean().sort_values(ascending=False).round(1))
ranking

zona       departamento_nombre
CABA       Comuna 1               10381.2
           Comuna 3                7499.8
           Comuna 4                6920.6
           Comuna 14               6154.8
           Comuna 9                5959.0
           Comuna 15               5526.9
           Comuna 7                5372.5
           Comuna 2                5266.4
           Comuna 8                4976.5
           Comuna 10               4797.1
           Comuna 5                4756.2
           Comuna 11               4371.7
           Comuna 13               4200.8
           Comuna 12               4082.1
           Comuna 6                4011.6
Conurbano  Morón                   3727.0
           General San Martín      3404.3
           Avellaneda              3315.0
           Ituzaingó               3241.4
           Tres de Febrero         3217.2
           Lanús                   2883.1
           Moreno                  2836.6
           San Isidro              2815.0
           Esteban Echeverría      2734.3
           Quilmes                 2731.3
           Malvinas Argentinas     2683.9
           José C. Paz             2681.1
           Tigre                   2635.2
           Berazategui             2619.6
           Hurlingham              2558.4
           Vicente López           2491.8
           San Miguel              2479.3
           Almirante Brown         2444.0
           Ezeiza                  2441.3
           San Fernando            2407.2
           Merlo                   2365.5
           Lomas de Zamora         2322.4
           La Matanza              2159.4
           Florencio Varela        2011.2
Name: tasa_total, dtype: float64

In [58]:
print(ranking.head(5))
print(ranking.tail(5))
print("\nPromedio simple por zona:")
print(ranking.groupby("zona").mean().round(1))

zona  departamento_nombre
CABA  Comuna 1               10381.2
      Comuna 3                7499.8
      Comuna 4                6920.6
      Comuna 14               6154.8
      Comuna 9                5959.0
Name: tasa_total, dtype: float64
zona       departamento_nombre
Conurbano  San Fernando           2407.2
           Merlo                  2365.5
           Lomas de Zamora        2322.4
           La Matanza             2159.4
           Florencio Varela       2011.2
Name: tasa_total, dtype: float64

Promedio simple por zona:
zona
CABA         5618.5
Conurbano    2716.9
Name: tasa_total, dtype: float64


**Interpretación:** al ajustar por población, el panorama se invierte respecto del volumen: aunque el conurbano registra más hechos en total, las comunas de CABA tienen tasas sensiblemente más altas. El promedio simple de la tasa total anual es de unos 5.620 hechos cada 100.000 habitantes en CABA y de unos 2.720 en el conurbano, es decir, algo más del doble. La diferencia es sistemática: la comuna con menor tasa (Comuna 6, cerca de 4.010) supera al partido con mayor tasa (Morón, cerca de 3.730).

Dentro de cada zona también hay dispersión. En CABA encabezan el ranking las comunas 1, 3, 4, 14 y 9, con la Comuna 1 muy por encima del resto (más de 10.000, casi el doble del promedio de CABA). En el conurbano, los partidos con menores tasas son Florencio Varela, La Matanza, Lomas de Zamora, Merlo y San Fernando (entre 2.000 y 2.400), mientras que Morón, General San Martín y Avellaneda se ubican en el extremo alto de la zona. La brecha entre el territorio de mayor y menor tasa es de unas 2,6 veces en CABA y de unas 1,9 en el conurbano.

Estos resultados deben leerse con cautela. La tasa usa como denominador la población residente, por lo que tiende a sobreestimarse en comunas céntricas con mucha población flotante, como la Comuna 1. Además, se trata de delitos registrados, que dependen de la propensión a denunciar y esta puede diferir entre territorios. Por último, los promedios son simples (no ponderados por población). La fuerte heterogeneidad entre comunas justifica profundizar, en la segunda parte del trabajo, en la relación entre el delito y las condiciones socioeconómicas dentro de CABA.